# מעבדה 02 — ריסון, תהודה ומקדם הטיב

במעבדה הזו תצפו בשלושת משטרי הריסון הופכים זה לזה, תמדדו עקומת תהודה כפי שנסיין עושה זאת —
על ידי סריקת הנעה והמתנה לגוויעת תגובת המעבר בכל נקודה — ואז תחלצו את מקדם הטיב $Q$ בשלוש
דרכים שונות לחלוטין מאותו מתנד, עם אי-ודאויות, ותבדקו שהן מתלכדות.

ניסוי אחד כאן נועד להפריך משהו שאתם כנראה מאמינים בו: שהתגובה מגיעה לשיא ב-$\omega_0$. היא
אינה.

עברו עליה לפי הסדר. במקום שבו המחברת מבקשת מכם לנבא, רשמו את הניבוי שלכם בתא המיועד **לפני**
הרצת התא הבא.

## מפרט המודל

| | |
|---|---|
| **מערכת** | מסת נקודה $m$ על קפיץ ליניארי $k$, עם ריסון מהירות $b$ והנעה $F_0\cos\omega t$ |
| **דינמיקה** | $m\ddot{x} + b\dot{x} + kx = F_0\cos\omega t$; צורות סגורות היכן שקיימות, velocity-Verlet במקומות האחרים |
| **שפה** | אין מרחבית; הסביבה היא בור חסר מאפיינים הבולע אנרגיה |
| **אנסמבל** | מסלול דטרמיניסטי אחד לכל מערכת פרמטרים; רעש המדידה גאוסי, בלתי תלוי ובעל זרע קבוע |
| **מוזנח** | המקור המיקרוסקופי של $b$, תגובת ההנעה חזרה, אי-ליניאריות של הקפיץ, חימום עצמי |
| **תקף כאשר** | ההזזות נותרות ליניאריות, הריסון צמיג באמת, טענות על המצב המתמיד ממתינות לגוויעת המעבר |
| **אופני כשל** | חיכוך יבש; משרעות תהודה היוצאות מהתחום הליניארי; מעבר הנקרא כמצב מתמיד |

כל הפיזיקה נמצאת ב-`wavelab.oscillators` וב-`wavelab.measurement` — פתחו אותם וקראו.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Those builds still have to be asked for by name:
# the kernel fetches a Pyodide package by itself only when a cell's own text imports it, and
# scipy is imported inside wavelab, where it does not look. Add any new compiled dependency
# of wavelab to the first list below and any new *pure-Python* one to the second.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["numpy", "scipy"])
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import measurement, oscillators
from wavelab.validation import seed_study

MASS = 0.5  # kg
STIFFNESS = 8.0  # N/m
OMEGA0 = oscillators.natural_frequency(MASS, STIFFNESS)
PERIOD = 2 * np.pi / OMEGA0
CRITICAL = 2 * np.sqrt(MASS * STIFFNESS)  # the damping that just kills the ringing

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"omega0 = {OMEGA0} rad/s   period = {PERIOD:.4f} s   critical b = {CRITICAL} kg/s")

## חלק 1 — שלושה משטרים, פונקציה אחת

שחררו את העגלה ממנוחה ותנו לה ללכת, בארבעה ערכי ריסון הפרוסים סביב הערך הקריטי. צפו במה
שמשתנה ב-$b = 2\sqrt{mk}$ — ובמה שאינו משתנה.

In [ ]:
t = np.linspace(0.0, 6 * PERIOD, 1500)
dampings = [0.1 * CRITICAL, 0.5 * CRITICAL, CRITICAL, 2.0 * CRITICAL]

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
for b in dampings:
    x = oscillators.damped_position(t, MASS, STIFFNESS, b, 0.1, 0.0)
    label = f"b/b_crit = {b / CRITICAL:.1f} ({oscillators.damping_regime(MASS, STIFFNESS, b)})"
    left.plot(t, x, lw=1.4, label=label)
    right.plot(x, np.gradient(x, t), lw=1.0)
left.axhline(0.0, color="0.85", lw=0.8, zorder=0)
left.set_xlabel("t (s)"), left.set_ylabel("x (m)"), left.legend(fontsize=8)
right.set_xlabel("x (m)"), right.set_ylabel("v (m/s)")
right.set_title("phase space: the ellipse unwinds")
plt.tight_layout()
plt.show()

# The ringing frequency is not omega0: damping slows it as well as killing it.
for b in dampings[:2]:
    gamma = oscillators.damping_rate(MASS, b)
    print(f"b/b_crit={b / CRITICAL:.2f}  omega_d = {np.sqrt(OMEGA0**2 - gamma**2 / 4):.4f} rad/s"
          f"   (omega0 = {OMEGA0:.4f})")

## חלק 2 — ניבוי

התחייבו לתשובות **לפני** הרצת התאים הבאים.

1. אתם סורקים את תדירות ההנעה ורושמים את משרעת המצב המתמיד. האם השיא מתחת ל-$\omega_0$,
   בדיוק בו, או מעליו?
2. אתם מניעים ממנוחה בדיוק ב-$\omega_0$ ולעולם לא מפסיקים. האם המשרעת גדלה ללא גבול? אם לא,
   מה עוצר אותה, וכמה זמן זה לוקח?
3. הכפלת הריסון מחצה את מספר המחזורים בדעיכה החופשית. מה היא עושה לרוחב שיא התהודה?

**הניבוי שלכם:**

## חלק 3 — Q מדעיכה חופשית

הראשונה מבין שלוש הדרכים ל-$Q$, והיחידה שחיה במרחב הזמן: הכו את המתנד פעם אחת וצפו בו גווע.
`q_from_ringdown` מתאים את מעטפת הדעיכה כדי לקבל את $\gamma$ וקורא את $\omega_d$ מחציות
האפס.

In [ ]:
DAMPING = 0.2  # kg/s  ->  Q = 10
GAMMA = oscillators.damping_rate(MASS, DAMPING)
TRUE_Q = oscillators.quality_factor(MASS, STIFFNESS, DAMPING)

ring_t = np.linspace(0.0, 20 * PERIOD, 8001)
ring_x = oscillators.damped_position(ring_t, MASS, STIFFNESS, DAMPING, 0.1, 0.0)
q_ringdown = oscillators.q_from_ringdown(ring_t, ring_x)

plt.figure(figsize=(9, 3.4))
plt.plot(ring_t, ring_x, lw=1.2, label="x(t)")
plt.plot(ring_t, 0.1 * np.exp(-GAMMA * ring_t / 2), "--", color="0.5", label="envelope e^(-γt/2)")
plt.plot(ring_t, -0.1 * np.exp(-GAMMA * ring_t / 2), "--", color="0.5")
plt.xlabel("t (s)"), plt.ylabel("x (m)"), plt.legend()
plt.title(f"ringdown: Q measured = {q_ringdown:.3f}, true = {TRUE_Q:.3f}")
plt.tight_layout()
plt.show()

## חלק 4 — עקומת התהודה, נמדדת

כעת מרחב התדירות. עבור כל אחת מ-30 תדירויות הנעה: התחילו ממנוחה, שלבו עד שתגובת המעבר מתה,
ומדדו את המשרעת והמופע ששרדו. זה מה שמדידה סרוקה *היא* — וההמתנה אינה אופציונלית, וזו
הנקודה של המשתנה `settle` שלהלן.

המשרעת והפיגור נקראים בעזרת נעילת מופע בת שתי שורות: הטילו את הזנב על $\cos\omega t$ ועל
$\sin\omega t$ לאורך מחזורים שלמים. כל מה שאינו בתדירות ההנעה מתאפס בממוצע.

In [ ]:
# Integrate from rest, wait out the transient, then lock in on the drive frequency.
def measure_steady_state(drive_omega, damping=DAMPING, force=1.0, steps_per_period=200):
    settle = 8.0 / oscillators.damping_rate(MASS, damping)  # about four transient lifetimes
    drive_period = 2 * np.pi / drive_omega
    dt = PERIOD / steps_per_period
    n_cycles = 8
    n_steps = int(round((settle + n_cycles * drive_period) / dt))
    run = oscillators.simulate(MASS, STIFFNESS, 0.0, 0.0, dt, n_steps,
                               damping=damping, drive_amplitude=force, drive_omega=drive_omega)

    keep = run.times >= run.times[-1] - n_cycles * drive_period
    t_tail, x_tail = run.times[keep], run.positions[keep]
    in_phase = 2 * np.mean(x_tail * np.cos(drive_omega * t_tail))
    quadrature = 2 * np.mean(x_tail * np.sin(drive_omega * t_tail))
    return np.hypot(in_phase, quadrature), np.arctan2(quadrature, in_phase)


sweep_omega = np.linspace(0.3 * OMEGA0, 1.8 * OMEGA0, 30)
measured = np.array([measure_steady_state(w) for w in sweep_omega])
predicted = oscillators.steady_state_response(sweep_omega, MASS, STIFFNESS, DAMPING, 1.0)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True)
top.plot(sweep_omega / OMEGA0, np.abs(predicted), lw=1.4, label="closed form |X|")
top.plot(sweep_omega / OMEGA0, measured[:, 0], "o", ms=4, label="integrated")
top.axvline(1.0, color="0.7", ls=":", label="omega0")
top.axvline(oscillators.resonance_peak_omega(MASS, STIFFNESS, DAMPING) / OMEGA0,
            color="green", ls="--", label="peak")
top.set_ylabel("|X| (m)"), top.legend(fontsize=8)
bottom.plot(sweep_omega / OMEGA0, np.angle(predicted), lw=1.4)
bottom.plot(sweep_omega / OMEGA0, measured[:, 1], "o", ms=4)
bottom.axhline(np.pi / 2, color="0.7", ls=":")
bottom.set_xlabel("omega / omega0"), bottom.set_ylabel("phase lag (rad)")
bottom.set_yticks([0, np.pi / 2, np.pi], ["0", "π/2", "π"])
plt.tight_layout()
plt.show()

worst = np.max(np.abs(measured[:, 0] - np.abs(predicted))) / np.max(np.abs(predicted))
print(f"largest amplitude discrepancy across the sweep: {worst:.2e} of the peak")

## חלק 5 — Q בשתי הדרכים האחרות

סריקה עדינה של הצורה הסגורה עומדת כעת במקום אחר צהריים סבלני של מדידות. ממנה, $Q$ לפי רוחב
הפס (הרוחב בחצי ההספק של עקומת המשרעת) ו-$Q$ לפי שיפוע המופע (כמה מהר הפיגור מסתובב דרך
$\pi/2$). יחד עם הדעיכה החופשית מחלק 3 אלה שלוש.

In [ ]:
fine_omega = np.linspace(0.5 * OMEGA0, 1.5 * OMEGA0, 4001)
fine = oscillators.steady_state_response(fine_omega, MASS, STIFFNESS, DAMPING, 1.0)

q_bandwidth = oscillators.q_from_bandwidth(fine_omega, np.abs(fine))
q_phase = oscillators.q_from_phase_slope(fine_omega, np.angle(fine))

print(f"true Q = sqrt(mk)/b            = {TRUE_Q:.4f}")
print(f"Q from the ringdown envelope   = {q_ringdown:.4f}   ({(q_ringdown/TRUE_Q - 1):+.2%})")
print(f"Q from the half-power bandwidth= {q_bandwidth:.4f}   ({(q_bandwidth/TRUE_Q - 1):+.2%})")
print(f"Q from the phase slope         = {q_phase:.4f}   ({(q_phase/TRUE_Q - 1):+.2%})")
print()
print("Three unrelated experiments, one number. The residuals are systematic, not scatter:")
print("the bandwidth route is low because the half-power width is only asymptotically gamma.")

## חלק 6 — הניסוי המפריך

ניבוי 1 אמר שהשיא נמצא ב-$\omega_0$. הנה המדידה שמכריעה: אתרו את המקסימום של עקומת המשרעת
נומרית, בערכי ריסון הפרוסים על פני שני סדרי גודל, ושרטטו את $\omega_\text{peak}/\omega_0$
מול $1/Q^2$.

אילו השיא באמת היה ב-$\omega_0$, כל נקודה הייתה יושבת על הקו האופקי ב-1.

In [ ]:
scan = np.linspace(1e-3, 1.6 * OMEGA0, 40_001)
test_dampings = np.geomspace(0.02, 3.0, 24)

inverse_q_sq, peak_ratio = [], []
for b in test_dampings:
    amp = np.abs(oscillators.steady_state_response(scan, MASS, STIFFNESS, b, 1.0))
    q = oscillators.quality_factor(MASS, STIFFNESS, b)
    inverse_q_sq.append(1 / q**2)
    peak_ratio.append(scan[np.argmax(amp)] / OMEGA0)

grid = np.linspace(0, 2.0, 200)
plt.figure(figsize=(8, 4))
plt.plot(inverse_q_sq, peak_ratio, "o", ms=5, label="measured maximum")
plt.plot(grid, np.sqrt(np.clip(1 - grid / 2, 0, None)), lw=1.4, label="sqrt(1 - 1/(2Q^2))")
plt.axhline(1.0, color="crimson", ls="--", lw=1.2, label="'the peak is at omega0'")
plt.axvline(2.0, color="0.6", ls=":", label="Q = 1/sqrt(2)")
plt.xlabel("1 / Q^2"), plt.ylabel("omega_peak / omega0"), plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

for q_target in (20.0, 2.0, 0.6):
    b = np.sqrt(MASS * STIFFNESS) / q_target
    peak_here = oscillators.resonance_peak_omega(MASS, STIFFNESS, b)
    verdict = f"{1 - peak_here / OMEGA0:7.4%} below omega0" if peak_here else "no peak at all"
    print(f"Q = {oscillators.quality_factor(MASS, STIFFNESS, b):6.2f}   peak: {verdict}")

הנקודות עוזבות את הקו האדום מיד ולעולם אינן חוזרות, ומעבר ל-$1/Q^2 = 2$ המקסימום הנמדד קורס
לאפס: השיא לא זז אל מחוץ לשרטוט, הוא חדל להתקיים. שימו לב גם *מדוע* התפיסה השגויה שורדת —
ב-$Q = 20$ ההיסט הוא $0.06\%$, קטן מכל קו שתוכלו לשרטט.

## חלק 7 — תגובת המעבר, וכמה זמן לוקחת תהודה

הניעו ממנוחה בדיוק ב-$\omega_0$. המעטפת מטפסת לעבר $Q F_0/k$ ונעצרת.

In [ ]:
dt = PERIOD / 200
build = oscillators.simulate(MASS, STIFFNESS, 0.0, 0.0, dt, 12_000,
                             damping=DAMPING, drive_amplitude=1.0, drive_omega=OMEGA0)
final = TRUE_Q * 1.0 / STIFFNESS
envelope = final * (1 - np.exp(-GAMMA * build.times / 2))

plt.figure(figsize=(9, 3.6))
plt.plot(build.times, build.positions, lw=0.9, label="x(t)")
plt.plot(build.times, envelope, "--", lw=1.4, color="0.4", label="Q (F0/k) (1 - e^(-γt/2))")
plt.axhline(final, color="green", ls=":", label=f"Q F0/k = {final:.3f} m")
plt.axhline(1.0 / STIFFNESS, color="crimson", ls=":", label=f"static F0/k = {1/STIFFNESS:.3f} m")
plt.xlabel("t (s)"), plt.ylabel("x (m)"), plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

reached = build.times[np.argmax(np.abs(build.positions) > 0.95 * final)]
print(f"1/e of the way up takes 2/gamma = {2/GAMMA/PERIOD:.2f} periods  "
      f"(Q/pi = {TRUE_Q/np.pi:.2f})")
print(f"95% of the way up takes {reached / PERIOD:.2f} periods  (three lifetimes: "
      f"{3*TRUE_Q/np.pi:.2f})")
print(f"amplification over the static response: {final * STIFFNESS:.2f}x  (Q = {TRUE_Q:.2f})")

## חלק 8 — מדידה, עם סרגל שגיאה

כעת התייחסו לציוד כאל אמיתי. הוסיפו רעש גלאי לרשומת הדעיכה ומדדו את $Q$ על פני שמונה זרעים
בלתי תלויים, בדיווח של ערך *וגם* אי-ודאות — לעולם לא מספר עירום.

In [ ]:
noise_sigma = 5e-4  # metres of detector noise on a 0.1 m release


def measure_q(generator):
    record = measurement.add_noise(ring_x, noise_sigma, generator)
    return oscillators.q_from_ringdown(ring_t, record)


study = seed_study(measure_q, n_seeds=8)
print(f"Q = {study.mean:.3f} ± {study.standard_error:.3f}   (true {TRUE_Q:.3f})")
print(f"relative spread across seeds: {study.relative_spread:.2%}")
print(f"agrees with the truth within 3 standard errors? {study.agrees_with(TRUE_Q, 3.0)}")

plt.figure(figsize=(8, 3.2))
plt.plot(ring_t, measurement.add_noise(ring_x, noise_sigma, np.random.default_rng(1)),
         lw=0.6, alpha=0.7, label="one noisy record")
plt.plot(ring_t, ring_x, lw=1.2, label="truth")
plt.xlabel("t (s)"), plt.ylabel("x (m)"), plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

נסו להעלות את `noise_sigma` פי עשרה ולהריץ מחדש. ההערכה אינה מתדרדרת בחן — היא קורסת, משום
שרעש סביב האפס שותל חציות מדומות וספירת חצאי המחזורים הופכת לחסרת משמעות. `q_from_ringdown`
מגן על עצמו בעזרת סף היסטרזיס; דחפו מעבר לו ותוכלו לצפות בהגנה נכשלת. לדעת היכן מכשיר מפסיק
לעבוד הוא חלק מהבעלות עליו.

## חלק 9 — בדיקות אוטומטיות

הטענות שהמחברת הזו העלתה, כקביעות ניתנות להרצה — אותן בדיקות רצות בחבילת הבדיקות של הפרויקט,
כך שהעמוד, המחברת והספרייה אינם יכולים להיפרד בשקט זה מזה.

In [ ]:
# 1. The measured sweep reproduces the closed form to better than 1% of the peak.
assert worst < 1e-2

# 2. The amplitude peak is strictly below omega0, and the power peak is exactly on it.
peak = oscillators.resonance_peak_omega(MASS, STIFFNESS, DAMPING)
assert peak < OMEGA0
power = oscillators.power_absorbed(scan, MASS, STIFFNESS, DAMPING, 1.0)
assert abs(scan[np.argmax(power)] - OMEGA0) < scan[1] - scan[0]

# 3. Below Q = 1/sqrt(2) the interior maximum does not exist at all.
assert oscillators.resonance_peak_omega(MASS, STIFFNESS, np.sqrt(2 * MASS * STIFFNESS)) == 0.0

# 4. The three routes to Q agree with the definition to better than 1%.
for name, value in (("ringdown", q_ringdown), ("bandwidth", q_bandwidth), ("phase", q_phase)):
    assert abs(value - TRUE_Q) / TRUE_Q < 1e-2, name

# 5. Steady state: the drive's cycle-averaged power equals what the damper dissipates.
tail = build.times >= build.times[-1] - 8 * PERIOD
input_power = np.mean(np.cos(OMEGA0 * build.times[tail]) * build.velocities[tail])
dissipated = np.mean(DAMPING * build.velocities[tail] ** 2)
assert np.isclose(input_power, dissipated, rtol=5e-3)

# 6. The noisy measurement recovers Q within its own uncertainty budget.
assert abs(study.mean - TRUE_Q) / TRUE_Q < 1e-2

print("all checks passed")
print(f"power in {input_power:.5f} W = power out {dissipated:.5f} W")

## חלק 10 — לחקור בעצמכם

המשלב רץ כאן חי, ולכן הפאנל הזה משתמש בכפתור **Run** (`interact_manual`): האינטגרציה איטית
מכדי לרוץ מחדש בכל תזוזה של מחוון בגרעין הדפדפן.

שלושה ניסויים שכדאי לעשות:

1. הורידו את $Q$ לעבר $0.707$ וצפו בשיא מחליק למטה ונעלם. היכן בדיוק עקומת המשרעת מפסיקה
   להיות בעלת מקסימום?
2. הסיטו מעט את ההנעה מ-$\omega_0$ ב-$Q$ גבוה וצפו בתגובת המעבר ובמצב המתמיד מכים זה בזה
   לפני שהמעבר מת.
3. העמידו את $Q$ על ערך גבוה מאוד וספרו כמה זמן אורכת הבנייה. בררנות נקנית בזמן.

In [ ]:
import ipywidgets as widgets


def explore(quality=10.0, detuning=1.0):
    damping = np.sqrt(MASS * STIFFNESS) / quality
    gamma = oscillators.damping_rate(MASS, damping)
    drive_omega = detuning * OMEGA0

    grid = np.linspace(0.2 * OMEGA0, 2.0 * OMEGA0, 2000)
    response = oscillators.steady_state_response(grid, MASS, STIFFNESS, damping, 1.0)
    step = PERIOD / 150
    run = oscillators.simulate(MASS, STIFFNESS, 0.0, 0.0, step, int(round(30 / gamma / step)),
                               damping=damping, drive_amplitude=1.0, drive_omega=drive_omega)

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
    left.plot(grid / OMEGA0, np.abs(response), lw=1.4)
    left.axvline(1.0, color="0.7", ls=":")
    left.axvline(detuning, color="crimson", lw=1.0)
    peak = oscillators.resonance_peak_omega(MASS, STIFFNESS, damping)
    if peak > 0:
        left.axvline(peak / OMEGA0, color="green", ls="--")
    left.set_xlabel("omega / omega0"), left.set_ylabel("|X| (m)"), left.set_title("response")
    right.plot(run.times, run.positions, lw=0.8)
    right.set_xlabel("t (s)"), right.set_ylabel("x (m)"), right.set_title("from rest")
    plt.tight_layout()
    plt.show()

    print(f"Q = {quality:.2f}   peak at {peak / OMEGA0 if peak else 0:.4f} omega0   "
          f"transient lifetime 2/gamma = {2 / gamma:.2f} s = {2 / gamma / PERIOD:.1f} periods")


widgets.interact_manual(
    explore,
    quality=widgets.FloatSlider(min=0.5, max=40.0, step=0.5, value=10.0, description="Q"),
    detuning=widgets.FloatSlider(min=0.3, max=1.8, step=0.02, value=1.0, description="ω/ω₀"),
);

## בדקו את הבנתכם

הריצו את התא שלהלן לקבלת הבוחן המדורג אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות,
נמצאות בעמוד המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "02-damped-driven.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת, בתא שלהלן.

1. איזה משלושת הניבויים שלכם היה שגוי, ומה בדיוק היה הפגם בנימוק שמאחוריו?
2. שלוש מדידות $Q$ הסכימו בדיוק טוב יותר מ-$1\%$ אך לא בדיוק מוחלט. נקבו בסיבה שיטתית אחת
   לשארית, ואמרו כיצד הייתם מבחינים בין שגיאה שיטתית לפיזור אקראי בעזרת הנתונים שבמחברת הזו
   בלבד.
3. עמית מדווח כי "תדירות התהודה של המערכת הזו היא $4.0$ רדיאן לשנייה". נקבו בשלושה גדלים
   שונים שאליהם ייתכן שהתכוון, ואמרו מתי משנה באיזה מהם מדובר.

**התשובות שלכם:**

1.
2.
3.